# Multilingual Narrative Understanding
SemEval-2025 Task 10 demonstration. Run from the repository root (or set ROOT below).
**No benchmark results are embedded in this notebook.** Obtain authorized data first; the small unit checks use invented examples only.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
ROOT = Path.cwd()
if ROOT.name == 'experiments':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('Repository:', ROOT)
subprocess.run([sys.executable, '-m', 'semeval', '--help'], check=True)

## 1. Check data and splits
Import official-format TSVs with the README commands. Repeat for all three subtasks, keeping held-out article IDs separate across tasks. Character offsets refer to unchanged Unicode text.

In [ ]:
paths = [ROOT / 'data' / 'st2' / f'{split}.jsonl' for split in ('train', 'dev', 'test')]
if all(path.exists() for path in paths):
    from semeval.data import read_jsonl, audit, check_splits
    datasets = [read_jsonl(path) for path in paths]
    check_splits(*datasets)
    print(json.dumps({str(path): audit(rows) for path, rows in zip(paths, datasets)}, indent=2, ensure_ascii=False))
else:
    print('Authorized data unavailable: import the corpus before running experiments.')

## 2. Classification experiments
Run `python -m experiments.run experiments/example.json` after data and the exact-release taxonomy are prepared. This compares TF-IDF, LSA, frozen MiniLM, and a label GCN, including graph ablations. For ST1, point a manifest at entity records; the graph is ST2-only.

In [ ]:
summary_path = ROOT / 'artifacts' / 'st2-comparison' / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    for run in summary['runs']:
        print(run['name'], run['primary_metric'], run['metrics'][run['primary_metric']], run['scoring_provenance'])
else:
    print('No measured classification runs available.')

## 3. Evidence and explanations
Use the `explain` command for TextRank, semantic extraction, and local generation. Keep gold-label and predicted-label results separate. Saved evidence IDs refer to the deterministic sentence splitter in `semeval.explain`; manual review must verify context and attribution.
The `export` command produces the official TSV layout. BERTScore configuration must be checked against the organizer scorer before any official comparison.

In [ ]:
from semeval.explain import rank_evidence, extract
invented = 'Invented example. Mira says the new policy will harm local businesses. Other speakers disagree.'
evidence = rank_evidence(invented, '', 'textrank')
print('Illustrative extraction only:', extract(evidence)[0])
print('This is not a benchmark prediction or finding.')

## 4. Present results honestly
Publish aggregate measured metrics only after checking split integrity and scorer provenance. The website initially displays an empty evidence board. Do not publish restricted article text, reference explanations, private predictions, or misleading synthetic scores.